In [1]:
import sys
import numpy as np
import anndata as ad

import polars as pl
import src as scit

/home/aleksander-work/Skrivbord/pyproject/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
cttag = ad.read_h5ad('private/data/clean_method/cttag_wt_inferred.h5ad')
cttag = cttag[cttag.obs['timep']=='14_16h'].copy()

In [3]:
me_class = pl.read_csv('private/newest/late_meth_cl_spec_lfc2.csv')
me_class

name,n,clusters
str,i64,str
"""14-3-3epsilon""",3,"""Fat Body;Midgut;Muscle"""
"""18w""",3,"""Epidermis;Fat Body;Muscle"""
"""4E-T""",2,"""Muscle;Ventral Nerve Cord"""
"""AANATL2""",3,"""Epidermis;Muscle;Ventral Nerve…"
"""AANATL4""",1,"""Ventral Nerve Cord"""
…,…,…
"""wor""",2,"""Epidermis;Muscle"""
"""z""",4,"""Epidermis;Fat Body;Muscle;Vent…"
"""zen""",5,"""Epidermis;Fat Body;Midgut;Musc…"


In [4]:
(me_class['n']==2).sum()

153

In [5]:
(me_class['n']==3).sum()

122

In [6]:
(me_class['n']==5).sum()

308

In [7]:
rna_change = pl.read_excel('private/data/snrna_difex_MAST_per_clusters_251117_all.xlsx', sheet_id=0)
rna_change.keys()

dict_keys(['Neuronal 1', 'Epidermis', 'Muscle somatic', 'Yolk', 'Midgut', 'Fat Body', 'Tracheal', 'Glia', 'Plasmatocytes', 'Muscle visceral', 'Neuronal 2', 'Salivary Gland', 'Epidermis head', 'Sense', 'Glia lateral', 'Amnioserosa', 'Bristle'])

In [8]:
#active_meth = pl.read_csv('private/data/meth_active.csv', has_header=False)['column_1'].to_numpy()
#muscle_up = pl.read_csv('private/muscle_meth_up.txt', has_header=False).to_numpy().ravel()
#muscle_down = pl.read_csv('private/muscle_meth_down.txt', has_header=False).to_numpy().ravel()

In [9]:
lc_acet = scit.tl.make_layer_config(
    "acet",
    in_obsm=True,
    log_transform=False,
    normalize_with_obs_counts=True,
    feature_names=cttag.uns['gene_names_acet']
)
lc_meth = scit.tl.make_layer_config(
    "meth",
    in_obsm=True,
    log_transform=False,
    normalize_with_obs_counts=True,
    feature_names=cttag.uns['gene_names_acet']
)

In [10]:
tf_df = pl.read_csv('private/fb_tfs.csv', has_header=False, 
            new_columns=['id', 'type', 'type_desc', 'id2', 'type2', 'id3', 'gene_name'])

In [11]:
tf_df=tf_df.with_columns(
    pl.when(pl.col('type2').is_null()).then(pl.lit('unknown')).otherwise(pl.col('type2')).alias('type2')
)

In [12]:
#mei = class_df.filter(pl.col('n') > 1, pl.col('n') < 5)['name'].to_numpy()
def get_clusters(_df, doprint: bool = False):
    clss = [me_class.filter(pl.col('n') == (i+1))['name'].to_numpy() for i in range(5)]
    dfs = [_df.filter(pl.col('gene_name').is_in(c)) for c in clss]
    if doprint:
        for i,d in enumerate(dfs):
            print(f"n={i+1} has {len(d['gene_name'].to_list())}")
            print(d['gene_name'].to_list())
    return dfs

tfs = get_clusters(tf_df)
cmap = {
    'BHLH': 'tab:green',
    'ZF-C2H2': 'tab:blue',
    'NKL': 'tab:purple',
    'ZFH': 'tab:cyan',
    'PRDH': 'tab:red',
    'PRDL': 'tab:orange'
}

In [23]:
# import matplotlib.pyplot as plt
# shapes = [me_class.filter(pl.col('n') == (i+1))['name'].shape[0] for i in range(5)]
# for i in range(5):
#     print(f'm. cluster n={i+1} (genes in cluster={shapes[i]}; total tfs={tfs[i].shape[0]})')
#     scit.pl.frequency_pie(tfs[i]['type'],color_map=cmap)
#     plt.gcf().savefig(f'private/newest/screenshots/pie{i+1}_type1.pdf')
#     plt.close()

In [24]:
import matplotlib.pyplot as plt
cmap = {
    'HTH': 'tab:blue',
    'TFS': 'tab:green',
    'ZNC4': 'tab:pink',
    'ZN-TF': 'tab:purple',
    'IG-TF': 'tab:brown',
    'BDTF': 'tab:cyan',
    'PRDTF': 'tab:red',
    'HBTF': 'tab:orange'
}
shapes = [me_class.filter(pl.col('n') == (i+1))['name'].shape[0] for i in range(5)]
for i in range(5):
    print(f'm. cluster n={i+1} (genes in cluster={shapes[i]}; total tfs={tfs[i].shape[0]})')
    scit.pl.frequency_pie(tfs[i]['type2'],color_map=cmap)
    # plt.gcf().savefig(f'private/newest/screenshots/piechart_{i+1}.pdf')
    plt.close()

m. cluster n=1 (genes in cluster=541; total tfs=70)
m. cluster n=2 (genes in cluster=153; total tfs=21)
m. cluster n=3 (genes in cluster=122; total tfs=24)
m. cluster n=4 (genes in cluster=161; total tfs=37)
m. cluster n=5 (genes in cluster=308; total tfs=105)


In [24]:
from xlsxwriter import Workbook
with Workbook('private/newest/supplementary-tables/pie-chart-cluster-meth.xlsx') as wb:  
    for i in range(5):
        tfs[i].write_excel(workbook=wb,worksheet=f'n{i}')

In [25]:
resmap = {k:set() for k in cmap.keys()}

for i in range(5):
    for k in cmap.keys():
        for u in tfs[i].filter(pl.col('type2')==k)['type_desc'].unique():
            resmap[k].add(u)

for k in resmap.keys():
    print(f"{k} = {resmap[k]}")

HTH = {'PIPSQUEAK TRANSCRIPTION FACTORS', 'FORK HEAD BOX TRANSCRIPTION FACTORS', 'A-T RICH INTERACTION DOMAIN TRANSCRIPTION FACTORS'}
TFS = {'ETS DOMAIN TRANSCRIPTION FACTORS', 'MADS-BOX TRANSCRIPTION FACTORS', 'UNCLASSIFIED DNA BINDING DOMAIN TRANSCRIPTION FACTORS', 'HIGH MOBILITY GROUP BOX TRANSCRIPTION FACTORS', 'GLIAL CELL MISSING TRANSCRIPTION FACTORS'}
ZNC4 = {'GATA TRANSCRIPTION FACTORS', 'NUCLEAR RECEPTOR (LIGAND-DEPENDENT) TRANSCRIPTION FACTORS', 'NUCLEAR RECEPTOR SUBFAMILY 0 (LIGAND-INDEPENDENT) TRANSCRIPTION FACTORS'}
ZN-TF = {'DMRT TRANSCRIPTION FACTORS', 'C2H2 ZINC FINGER TRANSCRIPTION FACTORS'}
IG-TF = {'T-BOX TRANSCRIPTION FACTORS', 'RUNT DOMAIN TRANSCRIPTION FACTORS'}
BDTF = {'BASIC HELIX-LOOP-HELIX TRANSCRIPTION FACTORS', 'BASIC LEUCINE ZIPPER TRANSCRIPTION FACTORS'}
PRDTF = {'PAIRED DOMAIN TRANSCRIPTION FACTORS', 'PAIRED HOMEOBOX TRANSCRIPTION FACTORS'}
HBTF = {'HOX-LIKE HOMEOBOX TRANSCRIPTION FACTORS', 'CUT HOMEOBOX TRANSCRIPTION FACTORS', 'SINE OCULIS HOMEOBOX TRANS